<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
.katex-display {
  text-align: left !important;
  margin: 1rem 0 1.25rem 2.5rem !important;
}
.katex-display > .katex {
  text-align: left !important;
}
.katex-display > .katex > .katex-html {
  text-align: left !important;
}
</style>

## 2-opt 알고리듬 (2-opt Algorithm)

<strong>2-opt 알고리듬은 경로의 두 연결을 바꾸어 총거리를 개선하는 지역 탐색 방법이다.</strong>

### 1 · 핵심 원리

- <strong>지역 탐색과의 관계:</strong> 지역 탐색은 현재 해의 일부를 바꾼 후보를 살펴보는 일반적인 탐색 방식이다. 2-opt는 경로의 두 연결을 바꾸는 구체적인 방법이다.
- <strong>2-opt(투 옵트):</strong> 두 연결을 교환하고 그 사이의 방문 순서를 뒤집어 이웃을 만든다.
- <strong>갱신·종료:</strong> 가장 짧은 이웃이 현재 경로보다 짧으면 갱신한다. 더 짧은 이웃이 없으면 종료한다.


<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
.katex-display {
  text-align: left !important;
  margin: 1rem 0 1.25rem 2.5rem !important;
}
.katex-display > .katex {
  text-align: left !important;
}
.katex-display > .katex > .katex-html {
  text-align: left !important;
}
</style>

### 2 · 적용 예시

- <strong>예시: 다섯 도시의 배송 경로 개선</strong>
  - 도시 0–4를 한 번씩 방문하고 도시 1로 복귀한다. 목적함수는 총거리이며, 고정된 대칭 거리는 코드의 `distance`에 주어진다.
  - 현재 경로 $(1,0,4,3,2,1)$의 거리는 22이다. 방문 구간 $(3,2)$를 뒤집으면 $(1,0,4,2,3,1)$이 된다.
  - 제거하는 두 연결의 거리는 $1+8=9$, 추가하는 거리는 $3+5=8$이다. 새 총거리는 $22-9+8=21$이다.
  - 아래에서 회색 연결을 제거하고 주황색 연결을 추가한다. 내부 구간은 방향만 바뀌므로 대칭 거리에서는 비용이 같다.

  <div style="text-align: left; margin: 0.65rem 0 1.5rem 2.5rem;"><img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/two_opt_reversal.svg" alt="위 경로의 구간 (3, 2)를 뒤집으면 아래 경로가 된다. 회색 연결 두 개를 제거하고 주황색 연결 두 개를 추가한다. 내부 구간의 비용 4는 그대로여서 총거리는 22에서 21로 줄어든다." width="440" style="display: block; width: 27.5em; max-width: 100%; height: auto; margin: 0;"></div>

### 3 · 구현

- `i`, `j`는 도시 번호가 아니라 경로 안의 위치이다. 시작·복귀 도시를 고정하고 그 사이 구간을 뒤집는다.
- 전체 역방향은 같은 순환 경로이므로 제외한다. 나머지 이웃을 모두 비교한 뒤 가장 짧은 경로로 갱신한다.


In [ ]:
distance = [
    [0, 2, 9, 10, 7],
    [2, 0, 8, 5, 6],
    [9, 8, 0, 4, 3],
    [10, 5, 4, 0, 1],
    [7, 6, 3, 1, 0],
]

def tour_cost(tour):
    total = 0
    for i in range(len(tour) - 1):
        a = tour[i]
        b = tour[i + 1]
        total += distance[a][b]
    return total

tour = [1, 0, 4, 3, 2, 1]
while True:
    current_distance = tour_cost(tour)
    best_distance = current_distance
    best_tour = tour

    for i in range(1, len(tour) - 2):
        for j in range(i + 1, len(tour) - 1):
            if i == 1 and j == len(tour) - 2:
                continue  # Skip the same tour in reverse.

            candidate = []
            # Keep the cities before the selected segment.
            for position in range(0, i):
                candidate.append(tour[position])
            # Add the selected segment in reverse order.
            for position in range(j, i - 1, -1):
                candidate.append(tour[position])
            # Keep the cities after the selected segment.
            for position in range(j + 1, len(tour)):
                candidate.append(tour[position])

            candidate_distance = tour_cost(candidate)
            if candidate_distance < best_distance:
                best_distance = candidate_distance
                best_tour = candidate

    if best_distance >= current_distance:
        break
    tour = best_tour

print("Route:", tour)
print("Distance:", tour_cost(tour))

<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
.katex-display {
  text-align: left !important;
  margin: 1rem 0 1.25rem 2.5rem !important;
}
.katex-display > .katex {
  text-align: left !important;
}
.katex-display > .katex > .katex-html {
  text-align: left !important;
}
</style>

- <strong>결과:</strong> `[1, 0, 2, 4, 3, 1]`. 총거리는 $22\to21\to20$으로 감소한다.
- <strong>종료 이유:</strong> 마지막 경로의 이웃 거리는 30·33·21·24·22이다. 현재 거리 20보다 짧은 이웃이 없다.
- <strong>한계:</strong> 선택한 이웃에 대한 지역 최적해일 뿐, 일반적으로 전역 최적성을 보장하지 않는다. 이 예제의 20이 최단거리인 것은 전체 경로 비교로 별도 확인할 수 있다.
